# Multimodal airline assistant

## Clients and database

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

azure = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
azure_model = os.getenv("GPT_MODEL", "gpt-4o")

import base64
import json
import sqlite3
import time
from io import BytesIO

import gradio as gr
from PIL import Image

DB = repo_root / "W2-Build Multi Model Chatbots" / "Day-5" / "airline.db"

with sqlite3.connect(DB) as conn:
    conn.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
    )
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS bookings (
            reference TEXT PRIMARY KEY,
            city TEXT,
            passenger_name TEXT,
            price REAL
        )
        """
    )

for city, price in {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999}.items():
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) "
            "ON CONFLICT(city) DO UPDATE SET price = excluded.price",
            (city, price),
        )


## Tools

Booking is the extra action. It refuses a city that has no fare, and it stores the passenger, city, price, and reference.

In [ ]:
def get_ticket_price(city):
    print(f"get_ticket_price({city})", flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city.lower(),),
        ).fetchone()
    if not row:
        return f"No price data available for {city}"
    return f"Ticket price to {city} is ${row[0]}"


def set_ticket_price(city, price):
    print(f"set_ticket_price({city}, {price})", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) "
            "ON CONFLICT(city) DO UPDATE SET price = excluded.price",
            (city.lower(), float(price)),
        )
    return f"Updated the ticket price to {city} to ${price}"


def book_flight(city, passenger_name):
    print(f"book_flight({passenger_name}, {city})", flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city.lower(),),
        ).fetchone()
        if not row:
            return f"No fare for {city}, so the flight was not booked."
        reference = f"FA{int(time.time()) % 100000}"
        conn.execute(
            "INSERT INTO bookings (reference, city, passenger_name, price) "
            "VALUES (?, ?, ?, ?)",
            (reference, city.lower(), passenger_name, row[0]),
        )
    return (
        f"Booked {passenger_name} to {city} for ${row[0]}. Reference {reference}."
    )


system_message = """
You are a helpful assistant for an airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
Use the tools to look up prices, update prices, and book flights.
Do not invent prices or booking references.
"""

city_property = {
    "type": "string",
    "description": "The city the customer wants to travel to",
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the price of a return ticket to the destination city.",
            "parameters": {
                "type": "object",
                "properties": {"destination_city": city_property},
                "required": ["destination_city"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_ticket_price",
            "description": "Set or update the return ticket price for a destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": city_property,
                    "price": {"type": "number", "description": "New return fare in USD"},
                },
                "required": ["destination_city", "price"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "book_flight",
            "description": "Book a return flight and store the booking.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": city_property,
                    "passenger_name": {
                        "type": "string",
                        "description": "Name of the passenger",
                    },
                },
                "required": ["destination_city", "passenger_name"],
                "additionalProperties": False,
            },
        },
    },
]


def handle_tool_calls(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        name = tool_call.function.name
        if name == "get_ticket_price":
            city = arguments["destination_city"]
            cities.append(city)
            content = get_ticket_price(city)
        elif name == "set_ticket_price":
            content = set_ticket_price(
                arguments["destination_city"],
                arguments["price"],
            )
        elif name == "book_flight":
            city = arguments["destination_city"]
            cities.append(city)
            content = book_flight(city, arguments["passenger_name"])
        else:
            content = f"Unknown tool {name}"
        responses.append(
            {
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            }
        )
    return responses, cities


## Image and speech

In [ ]:
def artist(city):
    image_model = os.getenv("IMAGE_MODEL")
    if not image_model:
        print("IMAGE_MODEL is not set, skipping the picture.")
        return None
    try:
        image_response = azure.images.generate(
            model=image_model,
            prompt=(
                f"A vibrant pop-art travel poster of {city}, "
                "showing well-known places in that city."
            ),
            size="1024x1024",
            n=1,
            timeout=120.0,
        )
        image = image_response.data[0]
        if image.b64_json:
            return Image.open(BytesIO(base64.b64decode(image.b64_json)))
        print("Image response had no b64_json payload.")
    except Exception as exc:
        print(f"Image generation failed: {exc}")
    return None


def talker(message):
    tts_model = os.getenv("TTS_MODEL")
    if not tts_model or not message:
        return None
    try:
        speech = azure.audio.speech.create(
            model=tts_model,
            voice="alloy",
            input=message,
            timeout=120.0,
        )
        path = repo_root / "W2-Build Multi Model Chatbots" / "Day-5" / "reply.mp3"
        path.write_bytes(speech.content)
        return str(path)
    except Exception as exc:
        print(f"Speech generation failed: {exc}")
        return None


## Blocks

Submit clears the box, appends the user turn, then `chat` fills the assistant turn, the audio player, and the image.

In [ ]:
def chat(history):
    history = [{"role": turn["role"], "content": turn["content"]} for turn in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = azure.chat.completions.create(
        model=azure_model,
        messages=messages,
        tools=tools,
        timeout=120.0,
    )
    cities = []

    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        tool_responses, found = handle_tool_calls(tool_message)
        cities.extend(found)
        messages.append(tool_message.model_dump(exclude_none=True))
        messages.extend(tool_responses)
        response = azure.chat.completions.create(
            model=azure_model,
            messages=messages,
            tools=tools,
            timeout=120.0,
        )

    reply = response.choices[0].message.content or ""
    history = history + [{"role": "assistant", "content": reply}]
    image = artist(cities[0]) if cities else None
    return history, talker(reply), image


def put_message_in_chatbot(message, history):
    history = history or []
    return "", history + [{"role": "user", "content": message}]


user = os.getenv("GRADIO_USER")
password = os.getenv("GRADIO_PASSWORD")
auth = (user, password) if user and password else None

with gr.Blocks(title="FlightAI") as ui:
    gr.Markdown("## FlightAI")
    with gr.Row():
        chatbot = gr.Chatbot(height=480)
        image_output = gr.Image(height=480, interactive=False)
    audio_output = gr.Audio(label="Spoken reply", autoplay=True)
    message = gr.Textbox(label="Chat with FlightAI")
    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output])

ui.launch(auth=auth)


Try `Price to Paris`, then `Book Akalanka to Paris`. A picture is requested only after a tool returns a city, and only if `IMAGE_MODEL` is set.